In [1]:
# Cài đặt các thư viện bắt buộc cho Pipeline và ép cập nhật torchao
!pip install -q -U "torchao>0.16.0"
!pip install -q open_clip_torch
!pip install -q evaluate
!pip install -q rouge_score
!pip install -q bert_score
!pip install -q datasets
!pip install -q accelerate
!pip install -q scikit-learn
!pip install -q transformers qwen-vl-utils
!pip install evaluate rouge_score bert_score scikit-learn
!pip install -q bitsandbytes accelerate
print("✅ Đã cài đặt xong tất cả các thư viện cần thiết và cập nhật torchao!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 31.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 53.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.4 MB/s eta 0:00:0000:0100:01
✅ Đã cài đặt xong tất cả các thư viện cần thiết và cập nhật torchao!


In [2]:
import os
import gc
import copy
import time
import torch
import re
import numpy as np
import pandas as pd
from PIL import Image
from datasets import load_dataset
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
import evaluate
from torch.optim.lr_scheduler import CosineAnnealingLR
import bitsandbytes as bnb 
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# ==========================================
# 0. THIẾT LẬP MÔI TRƯỜNG KAGGLE
# ==========================================
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

runName = "Centralized_VQARAD_Qwen2VL2B_LoRA_E2_DetailedLog"

kaggleBaseDir = "/kaggle/working/Fed-MVQA-Project"
dataDir = os.path.join(kaggleBaseDir, "data", "vqarad")
logDir = os.path.join(kaggleBaseDir, "logs")
checkpointDir = os.path.join(kaggleBaseDir, "checkpoints", runName)

os.makedirs(dataDir, exist_ok=True)
os.makedirs(logDir, exist_ok=True)
os.makedirs(checkpointDir, exist_ok=True)

logPath = os.path.join(logDir, f"{runName}_log.csv")
checkpointPath = os.path.join(checkpointDir, "best_centralized_model.pt")

# ==========================================
# 1. CÔNG CỤ ĐO LƯỜNG VÀ QUẢN LÝ BỘ NHỚ
# ==========================================
def clearMemory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def getPeakReservedMemory():
    if torch.cuda.is_available():
        # Lấy mức VRAM ĐỈNH ĐIỂM thực tế
        peak_gb = torch.cuda.max_memory_reserved() / (1024 ** 3)
        # Reset để đo chính xác cho chu kỳ tiếp theo
        torch.cuda.reset_peak_memory_stats()
        return f"{peak_gb:.2f} GB"
    return "0.00 GB"

def calculateTrainableParameters(model):
    trainableParams = 0
    allParams = 0
    for _, param in model.named_parameters():
        allParams += param.numel()
        if param.requires_grad:
            trainableParams += param.numel()
    percentage = 100 * trainableParams / allParams
    return trainableParams, allParams, percentage

def printStaticMemoryReport(totalParams, trainableParams):
    model_weights_gb = (totalParams * 2) / (1024**3)
    gradients_gb = (trainableParams * 2) / (1024**3)
    optimizer_states_gb = (trainableParams * 2) / (1024**3) # Dựa trên 8-bit Adam
    total_static_gb = model_weights_gb + gradients_gb + optimizer_states_gb
    
    print("\n[BÁO CÁO CẤP PHÁT BỘ NHỚ TĨNH (STATIC VRAM ALLOCATION)]")
    print(f"  + Trọng số Base Model (FP16) : {model_weights_gb:.4f} GB")
    print(f"  + Đạo hàm Gradients (FP16)   : {gradients_gb:.4f} GB")
    print(f"  + Optimizer (8-bit Adam)     : {optimizer_states_gb:.4f} GB")
    print(f"  => TỔNG VRAM TĨNH YÊU CẦU    : {total_static_gb:.4f} GB")
    return total_static_gb

# ==========================================
# 2. KHỞI TẠO MÔ HÌNH TOÀN CỤC (CENTRALIZED)
# ==========================================
print("--- KHỞI TẠO HẠ TẦNG CENTRALIZED LORA ---")
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    min_pixels=256 * 28 * 28,
    max_pixels=512 * 28 * 28 
)

baseModel = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto", 
    attn_implementation="sdpa"
)

baseModel.config.use_cache = False
baseModel.gradient_checkpointing_enable()

# TỐI ƯU HÓA 4: Tăng lora_dropout lên 0.1 để chống Overfitting trên tập VQA-RAD nhỏ
loraConfig = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.1, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
    bias="none"
)

centralizedModel = get_peft_model(baseModel, loraConfig)
trainableParams, totalModelParams, trainPct = calculateTrainableParameters(centralizedModel)
print(f"[Metrics] Trainable Params: {trainableParams:,} (~{trainPct:.3f}%)")

static_vram_gb = printStaticMemoryReport(totalModelParams, trainableParams)

# ==========================================
# 3. CHUẨN BỊ DỮ LIỆU VQA-RAD (GỘP CHUNG 1 TẬP)
# ==========================================
print("\n--- CHUẨN BỊ DỮ LIỆU VQA-RAD (FULL CENTRALIZED DATASET) ---")
dataset = load_dataset("flaviagiammarino/vqa-rad", cache_dir=dataDir)

trainDataset = dataset["train"].shuffle(seed=42)
testDataset = dataset["test"]

print(f"Dữ liệu Tập Train: {len(trainDataset)} mẫu")
print(f"Dữ liệu Tập Test: {len(testDataset)} mẫu")

headerTokens = processor.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)
headerLen = len(headerTokens)

# Ép mô hình trả lời 1-3 từ
systemInstruction = (
    "You are an expert medical diagnostic assistant. Your task is to answer clinical questions "
    "based EXCLUSIVELY on the provided medical image.\n"
    "Strictly follow these rules:\n"
    "1. Analyze the provided [Target Case Image] and the [Question].\n"
    "2. Provide a concise and definitive answer (1-3 words if possible)."
)

def customCollateFn(batch):
    texts = []
    images = []
    
    for item in batch:
        img = item["image"].convert("RGB")
        q = item["question"]
        a = str(item["answer"]).strip()
        
        messages = [
            {"role": "system", "content": [{"type": "text", "text": systemInstruction}]},
            {"role": "user", "content": [
                {"type": "text", "text": "[Target Case Image]\n"}, 
                {"type": "image", "image": img}, 
                {"type": "text", "text": f"\nCurrent Question: {q}\n\nBased on the visual evidence, what is the correct answer for the current case?\nFinal Answer:"}
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": a}]}
        ]
        
        textPrompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(textPrompt)
        images.append(img)

    inputs = processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = inputs["input_ids"].clone()
    
    for i in range(labels.shape[0]):
        tokenIds = labels[i].tolist()
        if processor.tokenizer.pad_token_id is not None:
            labels[i][labels[i] == processor.tokenizer.pad_token_id] = -100
            
        matchIdx = -1
        for j in range(len(tokenIds) - headerLen + 1):
            if tokenIds[j:j+headerLen] == headerTokens:
                matchIdx = j + headerLen 
                break
        if matchIdx != -1:
            labels[i, :matchIdx] = -100 

    inputs["labels"] = labels
    return inputs

batchSize = 2
trainLoader = DataLoader(trainDataset, batch_size=batchSize, shuffle=True, collate_fn=customCollateFn, num_workers=0, pin_memory=True)

# ==========================================
# 4. HÀM ĐÁNH GIÁ TÍCH HỢP ROBUST PARSING
# ==========================================
def cleanOpenAnswer(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    stopwords = {'a', 'an', 'the', 'is', 'are', 'was', 'were', 'it', 'there', 'shows', 'showing', 'of', 'in', 'on', 'to'}
    words = text.split()
    words = [w for w in words if w not in stopwords]
    return " ".join(words)

def evaluateModel(model, dataset, processorInput):
    model.eval()
    print(f"\n [Hệ thống] Đang đánh giá trên tập Test ({len(dataset)} mẫu)...")
    
    openPreds, openRefs = [], []
    closePreds, closeRefs = [], []
    overallCorrect = 0
    inferenceTimes = []
    
    yesKeywords = ['yes', 'correct', 'true', 'positive', 'present', 'visible']
    noKeywords = ['no', 'incorrect', 'false', 'negative', 'absent', 'normal', 'none']
    
    evalLoop = tqdm(dataset, desc="Evaluating", leave=False, dynamic_ncols=True)
    
    for item in evalLoop:
        img = item["image"].convert("RGB")
        q = item["question"]
        gtAnswer = str(item["answer"]).strip().lower()
        ansType = str(item.get("answer_type", "")).strip().upper()
        if ansType == "":
            ansType = "CLOSED" if gtAnswer in ["yes", "no"] else "OPEN"
        
        messages = [
            {"role": "system", "content": [{"type": "text", "text": systemInstruction}]},
            {"role": "user", "content": [
                {"type": "text", "text": "[Target Case Image]\n"}, 
                {"type": "image", "image": img}, 
                {"type": "text", "text": f"\nCurrent Question: {q}\n\nBased on the visual evidence, what is the correct answer for the current case?\nFinal Answer:"}
            ]}
        ]
        
        textPrompt = processorInput.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processorInput(text=[textPrompt], images=[img], return_tensors="pt", padding=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        startTime = time.time()
        with torch.no_grad():
            generatedIds = model.generate(**inputs, max_new_tokens=15, do_sample=False)
        inferenceTimes.append(time.time() - startTime)
            
        generatedIdsTrimmed = [outIds[len(inIds):] for inIds, outIds in zip(inputs["input_ids"], generatedIds)]
        aiAnswerRaw = processorInput.batch_decode(generatedIdsTrimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip().lower()
        
        aiAnswer = aiAnswerRaw.replace("final answer:", "").strip()
        
        if ansType == "CLOSED":
            hasYes = any(word in aiAnswer for word in yesKeywords)
            hasNo = any(word in aiAnswer for word in noKeywords)
            
            if hasYes and not hasNo:
                predClean = 'yes'
            elif hasNo and not hasYes:
                predClean = 'no'
            else:
                firstYesPos = aiAnswer.find('yes') if 'yes' in aiAnswer else 9999
                firstNoPos = aiAnswer.find('no') if 'no' in aiAnswer else 9999
                predClean = 'yes' if firstYesPos < firstNoPos else ('no' if firstNoPos < firstYesPos else 'unknown')
            
            gtClean = 'yes' if 'yes' in gtAnswer else ('no' if 'no' in gtAnswer else gtAnswer)
            
            closePreds.append(predClean)
            closeRefs.append(gtClean)
            
            if predClean == gtClean:
                overallCorrect += 1
        else:
            aiAnswerClean = cleanOpenAnswer(aiAnswer)
            gtAnswerClean = cleanOpenAnswer(gtAnswer)
            
            openPreds.append(aiAnswerClean)
            openRefs.append(gtAnswerClean)
            
            if aiAnswerClean == gtAnswerClean:
                overallCorrect += 1
                
        del inputs, generatedIds, generatedIdsTrimmed
        torch.cuda.empty_cache()

    results = {
        "overallAccuracy": overallCorrect / len(dataset) if len(dataset) > 0 else 0,
        "avgInferenceTime": sum(inferenceTimes) / len(inferenceTimes)
    }
    
    if len(closeRefs) > 0:
        acc = accuracy_score(closeRefs, closePreds)
        f1Weighted = f1_score(closeRefs, closePreds, average='weighted', zero_division=0)
        results["Closed"] = {"ACC": acc, "F1": f1Weighted, "count": len(closeRefs)}
    
    if len(openRefs) > 0:
        bleuMetric = evaluate.load("bleu")
        rougeMetric = evaluate.load("rouge")
        bertscoreMetric = evaluate.load("bertscore")
        
        formattedRefs = [[ref] for ref in openRefs] 
        bleuRes = bleuMetric.compute(predictions=openPreds, references=formattedRefs, max_order=1)
        rougeRes = rougeMetric.compute(predictions=openPreds, references=formattedRefs)
        bertRes = bertscoreMetric.compute(predictions=openPreds, references=openRefs, lang="en")
        
        results["Open"] = {
            "BLEU-1": bleuRes["bleu"],
            "ROUGE-L": rougeRes["rougeL"],
            "BERTScore": sum(bertRes["f1"]) / len(bertRes["f1"]),
            "count": len(openRefs)
        }
        
    return results

# ==========================================
# 5. VÒNG LẶP HUẤN LUYỆN TẬP TRUNG (CENTRALIZED)
# ==========================================
totalEpochs = 2 
learningRate = 5e-5
accumulationSteps = 4 

# Tối ưu bộ nhớ: Sử dụng PagedAdamW8bit thay vì AdamW chuẩn để có sự tương đồng 1:1 với Fed-LoRA
optimizer = bnb.optim.PagedAdamW8bit(centralizedModel.parameters(), lr=learningRate, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=totalEpochs * len(trainLoader) // accumulationSteps)

performanceLog = []
peakVramRecorded = "0.00 GB"
bestLoss = float('inf')

print("\n=== BẮT ĐẦU HUẤN LUYỆN TẬP TRUNG (CENTRALIZED LORA) ===")
startTotalTime = time.time()

for epoch in range(totalEpochs):
    centralizedModel.train()
    print(f"\n" + "="*50)
    print(f"--- EPOCH {epoch + 1}/{totalEpochs} ---")
    print("="*50)
    
    epochLoss = 0
    optimizer.zero_grad()
    
    progressLoop = tqdm(trainLoader, desc=f"Epoch {epoch + 1}/{totalEpochs}", leave=True, dynamic_ncols=True)
    
    for step, batch in enumerate(progressLoop):
        batch = {k: v.to(centralizedModel.device) for k, v in batch.items()}
        
        with torch.amp.autocast("cuda"):
            outputs = centralizedModel(**batch)
            loss = outputs.loss / accumulationSteps
        
        loss.backward()
        
        if (step + 1) % accumulationSteps == 0 or (step + 1) == len(trainLoader):
            torch.nn.utils.clip_grad_norm_(centralizedModel.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step() 
            optimizer.zero_grad()
            
        stepLoss = loss.item() * accumulationSteps
        epochLoss += stepLoss
        
        currentLr = scheduler.get_last_lr()[0]
        progressLoop.set_postfix(Loss=f"{stepLoss:.4f}", LR=f"{currentLr:.2e}")
        
        del outputs, loss, batch
            
    avgTrainLoss = epochLoss / len(trainLoader)
    peakVramRecorded = getPeakReservedMemory()
    clearMemory()
    
    # Đánh giá sau mỗi Epoch
    latestEvalResults = evaluateModel(centralizedModel, testDataset, processor)
    
    # Bóc tách Metric
    roundAccuracy = latestEvalResults["overallAccuracy"] * 100
    roundAccClosed = latestEvalResults.get("Closed", {}).get("ACC", 0) * 100
    roundF1Closed = latestEvalResults.get("Closed", {}).get("F1", 0) * 100
    roundBleu = latestEvalResults.get("Open", {}).get("BLEU-1", 0)
    roundRouge = latestEvalResults.get("Open", {}).get("ROUGE-L", 0)
    roundBert = latestEvalResults.get("Open", {}).get("BERTScore", 0)
    
    # Log vào mảng DataFrame
    performanceLog.append({
        "Epoch": epoch + 1,
        "Avg Loss": round(avgTrainLoss, 4),
        "Overall Acc (%)": round(roundAccuracy, 2),
        "Closed Acc (%)": round(roundAccClosed, 2),
        "Closed F1 (%)": round(roundF1Closed, 2),
        "Open BLEU-1": round(roundBleu, 4),
        "Open ROUGE-L": round(roundRouge, 4),
        "Open BERTScore": round(roundBert, 4),
        "Peak VRAM": peakVramRecorded
    })
    pd.DataFrame(performanceLog).to_csv(logPath, index=False)

    # Hiển thị kết quả đầy đủ ngay sau khi kết thúc 1 Epoch
    print(f"\n [KẾT QUẢ EPOCH {epoch + 1}]")
    print(f"  + Loss Trung Bình  : {avgTrainLoss:.4f}")
    print(f"  + Overall Accuracy : {roundAccuracy:.2f}%")
    print(f"  + Closed Questions : Acc = {roundAccClosed:.2f}%, F1 = {roundF1Closed:.2f}%")
    print(f"  + Open Questions   : BLEU-1 = {roundBleu:.4f}, ROUGE-L = {roundRouge:.4f}, BERTScore = {roundBert:.4f}")
    print(f"  + Mức sử dụng VRAM : Static = {static_vram_gb:.4f} GB | Dynamic Peak = {peakVramRecorded}")
    
    # Lưu Checkpoint
    if avgTrainLoss < bestLoss:
        bestLoss = avgTrainLoss
        torch.save(centralizedModel.state_dict(), checkpointPath)
        print(f"   -> [Cập nhật] Mô hình tốt nhất đã được lưu tại Epoch {epoch + 1}.")
    
    clearMemory()

# ==========================================
# 6. TỔNG HỢP VÀ IN BÁO CÁO 
# ==========================================
totalTime = time.time() - startTotalTime

print("\n" + "="*80)
print(" FINAL SUMMARY REPORT: CENTRALIZED BASELINE (VQA-RAD, LORA) ")
print("="*80)

print("\n[1] PERFORMANCE & HARDWARE METRICS")
print(f"  + Trainable Parameters   : {trainableParams:,} (~{trainPct:.3f}%)")
print(f"  + Total Training Time    : {totalTime:.2f} giây ({totalEpochs} Epochs)")
print(f"  + Static Allocated VRAM  : {static_vram_gb:.4f} GB")
print(f"  + Dynamic Peak VRAM      : {peakVramRecorded}")
print(f"  + Average Inference Time : {latestEvalResults['avgInferenceTime']:.4f} giây/ảnh")

print("\n[2] CLINICAL EVALUATION METRICS (TẠI EPOCH CUỐI)")
if "Closed" in latestEvalResults:
    print(f"  - CLOSED QUESTIONS ({latestEvalResults['Closed']['count']} samples):")
    print(f"    * Accuracy : {latestEvalResults['Closed']['ACC']*100:.2f}%")
    print(f"    * F1-Score : {latestEvalResults['Closed']['F1']*100:.2f}%")
if "Open" in latestEvalResults:
    print(f"  - OPEN QUESTIONS ({latestEvalResults['Open']['count']} samples):")
    print(f"    * BLEU-1   : {latestEvalResults['Open']['BLEU-1']:.4f}")
    print(f"    * ROUGE-L  : {latestEvalResults['Open']['ROUGE-L']:.4f}")
    print(f"    * BERTScore: {latestEvalResults['Open']['BERTScore']:.4f}")

print("\n[3] TRAINING LOG CHI TIẾT")
evalTable = pd.DataFrame(performanceLog)
print(evalTable.to_string(index=False))

print("\n" + "="*80)
print(" QUÁ TRÌNH HUẤN LUYỆN TẬP TRUNG ĐÃ HOÀN TẤT THÀNH CÔNG ")
print("="*80)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


--- KHỞI TẠO HẠ TẦNG CENTRALIZED LORA ---


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

... (31 outputs left out)

Evaluating:   0%|          | 0/451 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



 [KẾT QUẢ EPOCH 2]
  + Loss Trung Bình  : 0.4183
  + Overall Accuracy : 52.77%
  + Closed Questions : Acc = 72.91%, F1 = 72.97%
  + Open Questions   : BLEU-1 = 0.2636, ROUGE-L = 0.3624, BERTScore = 0.9028
  + Mức sử dụng VRAM : Static = 4.3209 GB | Dynamic Peak = 13.90 GB
   -> [Cập nhật] Mô hình tốt nhất đã được lưu tại Epoch 2.

 FINAL SUMMARY REPORT: CENTRALIZED BASELINE (VQA-RAD, LORA) 

[1] PERFORMANCE & HARDWARE METRICS
  + Trainable Parameters   : 36,929,536 (~1.644%)
  + Total Training Time    : 5598.95 giây (2 Epochs)
  + Static Allocated VRAM  : 4.3209 GB
  + Dynamic Peak VRAM      : 13.90 GB
  + Average Inference Time : 0.6150 giây/ảnh

[2] CLINICAL EVALUATION METRICS (TẠI EPOCH CUỐI)
  - CLOSED QUESTIONS (251 samples):
    * Accuracy : 72.91%
    * F1-Score : 72.97%
  - OPEN QUESTIONS (200 samples):
    * BLEU-1   : 0.2636
    * ROUGE-L  : 0.3624
    * BERTScore: 0.9028

[3] TRAINING LOG CHI TIẾT
 Epoch  Avg Loss  Overall Acc (%)  Closed Acc (%)  Closed F1 (%)  Open BLEU-1